In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# Load your dataset
df = pd.read_csv('Loan_default.csv')

# Separate features and target
X = df.drop(['LoanID', 'Default'], axis=1)
y = df['Default']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define preprocessor for numerical and categorical features
numeric_features = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 
                   'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio']
categorical_features = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']

# Make sure all features are in your dataset
numeric_features = [col for col in numeric_features if col in X.columns]
categorical_features = [col for col in categorical_features if col in X.columns]

preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), numeric_features),
                  ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

# Create and train pipeline with Random Forest
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train the model
model.fit(X_train, y_train)

# Save the trained model to disk
joblib.dump(model, 'loan_default_prediction_model.pkl')

# Evaluate model accuracy on test data
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy:.4f}")

# Optionally, print classification report and confusion matrix
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Function to take user input and predict
def predict_loan_default():
    # User input
    print("Please enter the following information to predict loan default:")
    age = float(input("Age: "))
    income = float(input("Income: "))
    loan_amount = float(input("Loan Amount: "))
    credit_score = float(input("Credit Score: "))
    months_employed = float(input("Months Employed: "))
    num_credit_lines = int(input("Number of Credit Lines: "))
    interest_rate = float(input("Interest Rate: "))
    loan_term = int(input("Loan Term (Months): "))
    dti_ratio = float(input("Debt-to-Income Ratio: "))
    education = input("Education (e.g., High School, Bachelor's, Master's, PhD): ")
    employment_type = input("Employment Type (e.g., Full-Time, Part-Time, Self-Employed): ")
    marital_status = input("Marital Status (e.g., Single, Married, Divorced): ")
    loan_purpose = input("Loan Purpose (e.g., Personal, Home, Education, Auto): ")
    has_mortgage = input("Do you have a mortgage? (Yes/No): ") == "Yes"
    has_dependents = input("Do you have dependents? (Yes/No): ") == "Yes"
    has_cosigner = input("Do you have a cosigner? (Yes/No): ") == "Yes"

    # Prepare input for prediction
    user_data = {
        'Age': [age],
        'Income': [income],
        'LoanAmount': [loan_amount],
        'CreditScore': [credit_score],
        'MonthsEmployed': [months_employed],
        'NumCreditLines': [num_credit_lines],
        'InterestRate': [interest_rate],
        'LoanTerm': [loan_term],
        'DTIRatio': [dti_ratio],
        'Education': [education],
        'EmploymentType': [employment_type],
        'MaritalStatus': [marital_status],
        'LoanPurpose': [loan_purpose],
        'HasMortgage': [has_mortgage],
        'HasDependents': [has_dependents],
        'HasCoSigner': [has_cosigner]
    }

    # Convert to DataFrame
    input_df = pd.DataFrame(user_data)

    # Preprocess the input data (similar to training data)
    input_df_processed = preprocessor.fit_transform(input_df)

    # Predict the loan default status
    prediction = model.predict(input_df_processed)
    prediction_prob = model.predict_proba(input_df_processed)[0][1]

    # Display result
    if prediction[0] == 1:
        print(f"Prediction: Default (Risk: {prediction_prob*100:.2f}%)")
    else:
        print(f"Prediction: No Default (Risk: {prediction_prob*100:.2f}%)")

# Call the function to make prediction
predict_loan_default()


Model accuracy: 0.8860

Classification Report:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45170
           1       0.61      0.04      0.07      5900

    accuracy                           0.89     51070
   macro avg       0.75      0.52      0.50     51070
weighted avg       0.86      0.89      0.84     51070


Confusion Matrix:
[[45034   136]
 [ 5684   216]]
Please enter the following information to predict loan default:


Age:  45
